### Mounting Google Drive

This cell mounts Google Drive in the Colab environment.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


### Utility functions for checking the panel structure

This cell defines functions for checking time coverage across securities.

`check_full_calendar` builds a complete monthly calendar between the minimum and maximum date in the input table. For each `secid`, the function counts the expected number of months, the actual number of observations, and the list of missing months. This checks whether the time series for a given security is continuous.

`get_panel_coverage_stats` evaluates the overall panel structure: the maximum number of years available for a single security, the maximum number of months within a year, and the distribution of observations by `secid`. This function is used for a quick diagnostic of how complete the initial features are.

`filter_no_gaps_gt2` keeps only securities without long gaps in their time series. The gap is computed from consecutive dates within each `secid`; if the distance between neighboring observations is too large, the security is excluded.

**Summary:** this cell does not construct features. It prepares panel quality-control tools. These functions are later applied to individual features and to the final table in order to check calendar completeness before modelling.


In [ ]:
import pandas as pd
def check_full_calendar(df, id_col="secid", date_col="month_end"):
    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    full_calendar = pd.date_range(
        data[date_col].min(),
        data[date_col].max(),
        freq="ME"
    )

    out = []

    for secid, g in data.groupby(id_col):
        actual = pd.DatetimeIndex(sorted(g[date_col].dropna().unique()))
        missing = full_calendar.difference(actual)

        out.append({
            id_col: secid,
            "expected_months": len(full_calendar),
            "actual_months": len(actual),
            "is_complete": len(missing) == 0,
            "missing_months": list(missing)
        })

    return pd.DataFrame(out)


def get_panel_coverage_stats(df, id_col="secid", date_col="month_end"):
    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    data["year"] = data[date_col].dt.year
    data["month"] = data[date_col].dt.month

    years_per_secid = (
        data.groupby(id_col)["year"]
        .nunique()
    )

    max_years = years_per_secid.max()

    months_per_year = (
        data.groupby([id_col, "year"])["month"]
        .nunique()
    )

    max_months_in_year = months_per_year.max()

    return {
        "max_years_across_secid": int(max_years),
        "max_months_in_a_year_across_secid": int(max_months_in_year),
        "years_per_secid": years_per_secid.sort_values(ascending=False),
        "months_per_year_per_secid": months_per_year.sort_values(ascending=False)
    }

import pandas as pd

def filter_no_gaps_gt2(df, col):
    """
    Оставляет строки только если в колонке col
    нет последовательности пропущенных месяцев длиной > 2.

    Иначе возвращает пустой DataFrame.
    """
    tmp = df.copy()
    tmp[col] = pd.to_datetime(tmp[col])
    tmp = tmp.sort_values(col)

    months = tmp[col].dt.to_period('M').drop_duplicates().sort_values()

    if len(months) <= 1:
        return tmp

    gaps = months[1:].astype(int).to_numpy() - months[:-1].astype(int).to_numpy()

    if (gaps > 3).any():
        return tmp.iloc[0:0]   

    return tmp


### Constructing the company size feature

This cell reads the monthly market capitalization table `monthly_market_cap.csv`, converts `month_end` to a date format, and keeps the capitalization, date, and security identifier fields.

The feature `size_it` is computed as the logarithm of market capitalization.

**Summary:** the output is the `size` table with a monthly size feature for each security. Rows with missing values are also printed in order to detect invalid capitalization values immediately. In this case, there is one such row.


In [ ]:
import pandas as pd
import numpy as np
import os
work_dir = "/content/drive/MyDrive/vega/LinearModels_kalman_comp/dataset_creating/raw_data"
df = pd.read_csv(os.path.join(work_dir, "monthly_market_cap.csv"), sep=";")
df['month_end'] = pd.to_datetime(df['month_end'])
mcap = df[['mcap_base_dict', 'month_end', 'secid']]
size = mcap.copy()
size["size_it"] = np.log(size["mcap_base_dict"])

print(f"non-valid: {size[size.isna().any(axis=1)].shape[0]}\n\n", size[size.isna().any(axis=1)], "\n\n")
size


non-valid: 1

        mcap_base_dict  month_end secid  size_it
17208             NaN 2019-01-31  MOBB      NaN 




/tmp/ipykernel_9487/3080612037.py:6: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['month_end'] = pd.to_datetime(df['month_end'])


,mcap_base_dict,month_end,secid,size_it
0,1.044114e+10,2023-07-31,ABIO,23.069020
1,1.033314e+10,2023-08-31,ABIO,23.058622
2,1.047226e+10,2023-09-30,ABIO,23.071996
3,9.864536e+09,2023-10-31,ABIO,23.012212
4,8.801019e+09,2023-11-30,ABIO,22.898133
...,...,...,...,...
38352,3.256909e+09,2023-03-31,ZVEZ,21.904044
38353,3.186656e+09,2023-04-30,ZVEZ,21.882238
38354,3.085492e+09,2023-05-31,ZVEZ,21.849977
38355,3.602551e+09,2023-06-30,ZVEZ,22.004908


### Checking coverage of the size feature

This cell applies the previously defined control functions to the `size` table.

First, aggregate panel characteristics are computed: the maximum number of years and months available by security. Then each `secid` is checked for the presence of observations in every month of the common calendar.

**Summary:** the cell shows how fully `size_it` is represented across securities and months. This check is needed before merging the size feature with the remaining variables.


In [ ]:
stats = get_panel_coverage_stats(size)

print(" total years across secid:", stats["max_years_across_secid"])
print(" total months across secid:", stats["max_months_in_a_year_across_secid"])

result = check_full_calendar(size)
print(f" valid secid: {result[result['actual_months']==result["expected_months"]].shape[0]}\n total secid: {result.shape[0]}")
result


 total years across secid: 14
 total months across secid: 12
 valid secid: 126
 total secid: 304


,secid,expected_months,actual_months,is_complete,missing_months
0,ABIO,168,30,False,"[2012-01-31 00:00:00, 2012-02-29 00:00:00, 201..."
1,ABRD,168,162,False,"[2012-01-31 00:00:00, 2012-02-29 00:00:00, 201..."
2,ACKO,168,10,False,"[2012-01-31 00:00:00, 2012-02-29 00:00:00, 201..."
3,AFKS,168,168,True,[]
4,AFLT,168,168,True,[]
...,...,...,...,...,...
299,YRSB,168,154,False,"[2014-07-31 00:00:00, 2014-08-31 00:00:00, 201..."
300,YRSBP,168,167,False,[2018-08-31 00:00:00]
301,ZAYM,168,21,False,"[2012-01-31 00:00:00, 2012-02-29 00:00:00, 201..."
302,ZILL,168,168,True,[]


### Constructing the company value feature

This cell reads the equity table `equity_monthly_from_quarters.csv`, converts monthly and quarterly dates to a date format, and merges book equity with market capitalization.

The feature `value_it` is computed as the logarithm of the ratio of book equity to market capitalization. If book equity is negative or the ratio is invalid, the feature value is set to missing; these missing values are allowed to be handled later at the training stage.

**Summary:** the output is the `value` table containing the value feature by security and month. The cell also prints the number of valid values and the date range.


In [ ]:
df = pd.read_csv(os.path.join(work_dir, "equity_monthly_from_quarters.csv"), sep=",")
df['month_end'] = pd.to_datetime(df['month_end'])
df['quarter_end'] = pd.to_datetime(df['quarter_end'])
equity = df[["secid", "month_end", "quarter_end", "equity"]]
value = mcap.merge(equity, on=["month_end", "secid"], how="inner")
value["value_it"] = np.log(value["equity"] / value["mcap_base_dict"])

print(f"valid value: {value["value_it"].dropna().shape[0]}\n")
print(value['month_end'].min(), value['month_end'].max(), "\n\n")
value


valid value: 11999

2016-10-31 00:00:00 2025-12-31 00:00:00 




/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,mcap_base_dict,month_end,secid,quarter_end,equity,value_it
0,1.044114e+10,2023-07-31,ABIO,2023-09-30,5.628253e+09,-0.617955
1,1.033314e+10,2023-08-31,ABIO,2023-09-30,5.628253e+09,-0.607557
2,1.047226e+10,2023-09-30,ABIO,2023-09-30,5.628253e+09,-0.620931
3,9.864536e+09,2023-10-31,ABIO,2023-12-31,5.452935e+09,-0.592792
4,8.801019e+09,2023-11-30,ABIO,2023-12-31,5.452935e+09,-0.478714
...,...,...,...,...,...,...
12692,2.079476e+09,2022-12-31,ZVEZ,2022-12-31,-1.364299e+09,NaN
12693,3.186656e+09,2023-04-30,ZVEZ,2023-06-30,-1.653931e+09,NaN
12694,3.085492e+09,2023-05-31,ZVEZ,2023-06-30,-1.653931e+09,NaN
12695,3.602551e+09,2023-06-30,ZVEZ,2023-06-30,-1.653931e+09,NaN


### Constructing illiquidity and the quarterly target variable

This cell reads `illiq_monthly.xlsx`. From this file, the monthly illiquidity measure `illiq_amihud_m` is extracted and then used as one of the model factors.

Next, the quarterly forward target is constructed from monthly returns. For each security, returns over the next three months are taken, calendar continuity is checked, and the compounded return over the future quarter is computed. The target is then cleaned from extreme values by winsorizing the lower and upper percentiles.

The variable `target_q_fwd_clean` is the dependent variable in the quarterly forecasting setup: at date `t`, the model uses available features and predicts the return over the next quarter.

**Summary:** this cell prepares both the illiquidity feature and the target variable `y`. It is the key transition from a monthly feature panel to a quarterly forecasting object.


In [ ]:
kef = pd.read_excel(os.path.join(work_dir, "illiq_monthly.xlsx"))
illiq_it = kef[["secid", "month_end", "illiq_amihud_m"]]


import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd

kef = kef[["ret_m", "month_end", "secid"]]
df = kef.copy()

df["month_end"] = pd.to_datetime(df["month_end"])


df = df.sort_values(["secid", "month_end"]).reset_index(drop=True)

g = df.groupby("secid", group_keys=False)

df["ret_t1"] = g["ret_m"].shift(-1)
df["ret_t2"] = g["ret_m"].shift(-2)
df["ret_t3"] = g["ret_m"].shift(-3)

df["date_t1"] = g["month_end"].shift(-1)
df["date_t2"] = g["month_end"].shift(-2)
df["date_t3"] = g["month_end"].shift(-3)

valid_3m_window = (
    df["date_t1"].eq(df["month_end"] + MonthEnd(1)) &
    df["date_t2"].eq(df["month_end"] + MonthEnd(2)) &
    df["date_t3"].eq(df["month_end"] + MonthEnd(3))
)

df["target_q_fwd_raw"] = np.where(
    valid_3m_window,
    (1 + df["ret_t1"]) * (1 + df["ret_t2"]) * (1 + df["ret_t3"]) - 1,
    np.nan
)

q01 = df["target_q_fwd_raw"].quantile(0.01)
q99 = df["target_q_fwd_raw"].quantile(0.99)

df["target_q_fwd_clean"] = df["target_q_fwd_raw"].clip(lower=q01, upper=q99)


df = df.drop(columns=[
    "ret_t1", "ret_t2", "ret_t3",
    "date_t1", "date_t2", "date_t3"
])

y = df

df = pd.read_excel(os.path.join(work_dir, "illiq_monthly.xlsx"))
illiq_it = df[["secid", "month_end", "illiq_amihud_m"]]

illiq_it['month_end'] = pd.to_datetime(illiq_it['month_end'])
illiq_it


/tmp/ipykernel_9487/1513068491.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  illiq_it['month_end'] = pd.to_datetime(illiq_it['month_end'])


,secid,month_end,illiq_amihud_m
0,ABIO,2009-12-31,1.134537e-09
1,ABIO,2010-01-31,1.413304e-09
2,ABIO,2010-02-28,2.729674e-09
3,ABIO,2010-03-31,1.278595e-09
4,ABIO,2010-04-30,1.480389e-09
...,...,...,...
52214,ZVEZ,2023-03-31,3.307306e-09
52215,ZVEZ,2023-04-30,3.439684e-09
52216,ZVEZ,2023-05-31,1.805905e-08
52217,ZVEZ,2023-06-30,8.233723e-09


### Checking the structure of the illiquidity feature

This cell prints information about the `illiq_it` table: the number of rows, column types, and the number of non-null values.


In [ ]:
illiq_it.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52219 entries, 0 to 52218
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   secid           52219 non-null  object        
 1   month_end       52219 non-null  datetime64[ns]
 2   illiq_amihud_m  51592 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 1.2+ MB


### Loading market and regime features

This cell reads `market_block1a.xlsx` and extracts features that are common to all securities in a given month: momentum, market volatility, the high-inflation indicator, and the drawdown-stress indicator.

`MOM` and `mkt_vol` are used as factor variables. `state_infl_high` and `state_dd_stress` define observable market regimes, which later enter the model directly and through interactions with security-level features.

**Summary:** the output consists of separate monthly market-feature tables, which are then joined to the panel data by `month_end`.


In [ ]:
df = pd.read_excel(os.path.join(work_dir, "market_block1a.xlsx"))
df['month_end'] = pd.to_datetime(df['month_end'])

mom_12_1 = df[["MOM", "month_end"]]
vol_12 = df[["mkt_vol", "month_end"]]
state_infl_high = df[["state_infl_high", "month_end"]]
state_dd_stress = df[["state_dd_stress", "month_end"]]

mom_12_1


,MOM,month_end
0,0.037539,2014-07-31
1,-0.012618,2014-08-31
2,-0.139195,2014-09-30
3,0.033327,2014-10-31
4,0.019627,2014-11-30
...,...,...
130,0.059378,2025-05-31
131,0.002282,2025-06-30
132,-0.000572,2025-07-31
133,-0.221878,2025-08-31


### Constructing regime interactions

This cell constructs two interactions between security-level features and regime indicators.

`value_x_inflhigh` multiplies the value feature of a security by the high-inflation indicator.

`illiq_x_stress` multiplies the illiquidity feature of a security by the drawdown-stress indicator.


In [ ]:
value_x_inflhigh = (
    value[["value_it", "month_end", "secid"]].merge(state_infl_high, on='month_end', how='left', suffixes=('_sector', '_date'))
       .assign(value_x_inflhigh=lambda x: x['value_it'] * x['state_infl_high'])
)

illiq_x_stress = (
    illiq_it[["illiq_amihud_m", "month_end", "secid"]].merge(state_dd_stress, on='month_end', how='left', suffixes=('_sector', '_date'))
       .assign(illiq_x_stress=lambda x: x['illiq_amihud_m'] * x['state_dd_stress'])
)
value_x_inflhigh


,value_it,month_end,secid,state_infl_high,value_x_inflhigh
0,-0.617955,2023-07-31,ABIO,0.0,-0.000000
1,-0.607557,2023-08-31,ABIO,0.0,-0.000000
2,-0.620931,2023-09-30,ABIO,1.0,-0.620931
3,-0.592792,2023-10-31,ABIO,1.0,-0.592792
4,-0.478714,2023-11-30,ABIO,1.0,-0.478714
...,...,...,...,...,...
12692,NaN,2022-12-31,ZVEZ,1.0,NaN
12693,NaN,2023-04-30,ZVEZ,0.0,NaN
12694,NaN,2023-05-31,ZVEZ,0.0,NaN
12695,NaN,2023-06-30,ZVEZ,0.0,NaN


### List of tables to merge

This cell collects all prepared components into the list `dfs`: size, value, momentum, volatility, illiquidity, two regime indicators, two interaction terms, and the quarterly target variable.

**Summary:** this cell defines the composition of the final panel. The actual merge is performed in the next cell; here, the exact set of features included in the dataset is fixed.


In [ ]:
from functools import reduce
import pandas as pd

dfs = [
    size[["size_it", "month_end", "secid"]],
    value[["value_it", "month_end", "secid"]],
    mom_12_1[["MOM", "month_end"]],
    vol_12[["mkt_vol", "month_end"]],
    illiq_it[["illiq_amihud_m", "month_end", "secid"]],
    state_infl_high[["state_infl_high", "month_end"]],
    state_dd_stress[["state_dd_stress", "month_end"]],
    value_x_inflhigh[["value_x_inflhigh", "month_end", "secid"]],
    illiq_x_stress[["illiq_x_stress", "month_end", "secid"]],
    y[["target_q_fwd_clean", "month_end", "secid"]]
]


### Building the final feature and target panel

This cell sequentially merges the tables from `dfs`. Security-level features are joined by the pair `month_end`, `secid`, while market and regime features are joined by `month_end`, since they are the same for all securities in a given month.

After the merge, the working date range is set from October 2016 to September 2025. The rows are then sorted by security and month, and the result is saved to `data.csv`.

**Summary:** this is the main dataset construction cell. The output is a unified panel `data_with_secid`, which contains features, regimes, interactions, and the quarterly target variable for subsequent modelling.


In [ ]:
data_with_secid = (
    dfs[0]
    .merge(dfs[1], on=["month_end", "secid"], how="outer")
    .merge(dfs[4], on=["month_end", "secid"], how="outer")
    .merge(dfs[7], on=["month_end", "secid"], how="outer")
    .merge(dfs[8], on=["month_end", "secid"], how="outer")
    .merge(dfs[2], on=["month_end"], how="outer") # month data
    .merge(dfs[3], on=["month_end"], how="outer") # month data
    .merge(dfs[5], on=["month_end"], how="outer") # month data
    .merge(dfs[6], on=["month_end"], how="outer") # month data
    .merge(dfs[9], on=["month_end", "secid"], how="outer") # month data
)

data_with_secid = data_with_secid[ # dataframe from market_block1a.xlsx
    (data_with_secid["month_end"] >= pd.Timestamp("2016-10-31")) &
    (data_with_secid["month_end"] <= pd.Timestamp("2025-09-30"))
]

data_with_secid = data_with_secid[
    data_with_secid['size_it'].notna() & data_with_secid['illiq_amihud_m'].notna() & data_with_secid['target_q_fwd_clean'].notna()
]
data_with_secid.to_csv("/content/drive/MyDrive/vega/data/data.csv", index=False)
data_with_secid


,size_it,month_end,secid,value_it,illiq_amihud_m,value_x_inflhigh,illiq_x_stress,MOM,mkt_vol,state_infl_high,state_dd_stress,target_q_fwd_clean
24434,22.709715,2016-10-31,ABRD,-1.192200,1.843932e-07,-0.0,0.000000e+00,0.038509,0.068871,0.0,0.0,0.266088
24435,25.739235,2016-10-31,AFKS,0.491660,9.866532e-11,0.0,0.000000e+00,0.038509,0.068871,0.0,0.0,0.240208
24436,25.407146,2016-10-31,AFLT,-0.296973,3.951919e-11,-0.0,0.000000e+00,0.038509,0.068871,0.0,0.0,0.340285
24438,25.094363,2016-10-31,AKRN,-1.033306,6.673845e-10,-0.0,0.000000e+00,0.038509,0.068871,0.0,0.0,0.170000
24439,21.415803,2016-10-31,ALBK,NaN,2.199913e-05,NaN,0.000000e+00,0.038509,0.068871,0.0,0.0,-0.175127
...,...,...,...,...,...,...,...,...,...,...,...,...
50996,20.118826,2025-09-30,YKENP,NaN,8.055373e-07,NaN,8.055373e-07,0.092241,0.229734,0.0,1.0,-0.178049
50997,23.376148,2025-09-30,YRSB,-2.490154,4.388706e-07,-0.0,4.388706e-07,0.092241,0.229734,0.0,1.0,-0.024155
50998,20.560311,2025-09-30,YRSBP,0.325682,8.823363e-07,0.0,8.823363e-07,0.092241,0.229734,0.0,1.0,-0.015915
50999,23.370150,2025-09-30,ZAYM,NaN,4.097550e-09,NaN,4.097550e-09,0.092241,0.229734,0.0,1.0,0.095322


In [ ]:
data_with_secid.info()


<class 'pandas.core.frame.DataFrame'>
Index: 24636 entries, 24434 to 51000
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   size_it             24636 non-null  float64       
 1   month_end           24636 non-null  datetime64[ns]
 2   secid               24636 non-null  object        
 3   value_it            11863 non-null  float64       
 4   illiq_amihud_m      24636 non-null  float64       
 5   value_x_inflhigh    11863 non-null  float64       
 6   illiq_x_stress      24636 non-null  float64       
 7   MOM                 24636 non-null  float64       
 8   mkt_vol             24636 non-null  float64       
 9   state_infl_high     24636 non-null  float64       
 10  state_dd_stress     24636 non-null  float64       
 11  target_q_fwd_clean  24636 non-null  float64       
dtypes: datetime64[ns](1), float64(10), object(1)
memory usage: 2.4+ MB


### Counting missing values in key variables

This cell groups the final table by `secid` and counts missing values in the main columns: `size_it`, `value_it`, `illiq_amihud_m`, and `target_q_fwd_clean`.

**Summary:** the result shows which securities have coverage problems in the base factors and the target variable. This diagnostic is needed before model training.


In [ ]:
cols = ['size_it', 'value_it', 'illiq_amihud_m', 'target_q_fwd_clean']

miss_count = data_with_secid.groupby('secid')[cols].apply(lambda x: x.isna().sum())
miss_count


,size_it,value_it,illiq_amihud_m,target_q_fwd_clean
secid,,,,
ABIO,81,84,0,0
ABRD,0,42,0,0
ACKO,3,13,0,3
AFKS,0,45,0,0
AFLT,0,78,0,0
...,...,...,...,...
YRSB,0,40,0,3
YRSBP,0,41,0,3
ZAYM,0,18,0,0


### Checking calendar completeness of the final panel

This cell reapplies the calendar control functions to the full table `data_with_secid`.

First, general time-coverage characteristics are printed. Then each `secid` is checked for the presence of observations in all months of the working date range.

**Summary:** the check is applied not to an individual factor, but to the final modelling panel. This makes it possible to understand how many securities can be used without substantial gaps in their time series.


In [ ]:
stats = get_panel_coverage_stats(data_with_secid)

print(" total years across secid:", stats["max_years_across_secid"])
print(" total months across secid:", stats["max_months_in_a_year_across_secid"])

result = check_full_calendar(data_with_secid)
print(f" valid secid: {result[result['actual_months']==result["expected_months"]].shape[0]}\n total secid: {result.shape[0]}")
result


 total years across secid: 10
 total months across secid: 12
 valid secid: 172
 total secid: 290


,secid,expected_months,actual_months,is_complete,missing_months
0,ABIO,108,27,False,"[2016-10-31 00:00:00, 2016-11-30 00:00:00, 201..."
1,ABRD,108,108,True,[]
2,ACKO,108,10,False,"[2016-10-31 00:00:00, 2016-11-30 00:00:00, 201..."
3,AFKS,108,108,True,[]
4,AFLT,108,108,True,[]
...,...,...,...,...,...
285,YRSB,108,103,False,"[2017-04-30 00:00:00, 2017-05-31 00:00:00, 201..."
286,YRSBP,108,104,False,"[2018-05-31 00:00:00, 2018-06-30 00:00:00, 201..."
287,ZAYM,108,18,False,"[2016-10-31 00:00:00, 2016-11-30 00:00:00, 201..."
288,ZILL,108,108,True,[]


In [ ]:
filter_no_gaps_gt2(data_with_secid, "month_end").groupby(by="secid").count()

,size_it,month_end,value_it,illiq_amihud_m,value_x_inflhigh,illiq_x_stress,MOM,mkt_vol,state_infl_high,state_dd_stress,target_q_fwd_clean
secid,,,,,,,,,,,
ABIO,27,27,24,27,24,27,27,27,27,27,27
ABRD,108,108,66,108,66,108,108,108,108,108,108
ACKO,10,10,0,10,0,10,10,10,10,10,10
AFKS,108,108,63,108,63,108,108,108,108,108,108
AFLT,108,108,30,108,30,108,108,108,108,108,108
...,...,...,...,...,...,...,...,...,...,...,...
YRSB,103,103,66,103,66,103,103,103,103,103,103
YRSBP,104,104,66,104,66,104,104,104,104,104,104
ZAYM,18,18,0,18,0,18,18,18,18,18,18
